# Lib


In [ ]:
import copy
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    resnet34,
    resnet50,
    ResNet34_Weights,
    ResNet50_Weights,
)

from scipy.ndimage import binary_erosion, distance_transform_edt

import kagglehub
from typing import List, Sequence, Tuple
import torch.utils.checkpoint as checkpoint

try:
    from timm.layers import DropPath, to_2tuple, trunc_normal_
except ImportError:
    from timm.models.layers import DropPath, to_2tuple, trunc_normal_

# SwinV2-Unet architecture


In [ ]:
DEFAULT_EVAL_CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "image_height": 256,
    "image_width": 256,
    "num_classes": 3,
    # SwinV2-Unet architecture: must match training checkpoint.
    "backbone": "swinv2_tiny_patch4_window8_256",
    "img_size": 256,
    "patch_size": 4,
    "in_channels": 3,
    "embed_dim": 96,
    "depths": (2, 2, 6, 2),
    "num_heads": (3, 6, 12, 24),
    "window_size": 8,
    "mlp_ratio": 4.0,
    "qkv_bias": True,
    "drop_rate": 0.0,
    "attn_drop_rate": 0.0,
    "drop_path_rate": 0.2,
    "ape": False,
    "patch_norm": True,
    "use_checkpoint": False,
    "pretrained_window_sizes": (0, 0, 0, 0),
    # Evaluation data.
    "batch_size": 16,
    # Keep this at 0 in a Windows/Jupyter notebook. Dataset classes defined in a
    # notebook cell are not reliably importable by spawned DataLoader workers.
    "num_workers": 0,
    "image_col": None,
    "mask_col": None,
    "dataset_handle": "avcuongy/bkai-igh-neopolyp-small-sample",
    # Input preprocessing.
    "normalize_input": True,
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    # Checkpoint loading.
    "strict_weight_loading": True,
}

BASE_CLASS_NAMES = {
    0: "background",
    1: "non-neoplastic",
    2: "neoplastic",
}

In [ ]:
def decode_ground_truth(mask_rgb):
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],
            [0, 255, 0],
            [255, 0, 0],
        ],
        dtype=np.float32,
    )

    distances = np.sum(
        (mask.astype(np.float32)[:, :, None, :] - colors[None, None, :, :]) ** 2,
        axis=-1,
    )

    return np.argmin(distances, axis=-1).astype(np.uint8)


def pick_column(
    dataframe,
    explicit_column=None,
    candidates=None,
    required=True,
):
    if explicit_column is not None:
        if explicit_column not in dataframe.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {dataframe.columns.tolist()}"
            )
        return explicit_column

    normalized = {str(column).strip().lower(): column for column in dataframe.columns}

    for candidate in candidates or []:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(dataframe.columns) == 1:
        return dataframe.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available: {dataframe.columns.tolist()}"
        )

    return None


class SegmentationEvaluationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image",
        "image_name",
        "image_path",
        "filename",
        "file_name",
        "file",
        "id",
    ]

    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        transform=None,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform

        self.image_col = pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

        self.mask_col = pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=False,
        )

    @staticmethod
    def resolve_path(root_dir, value):
        path = Path(str(value))
        return path if path.is_absolute() else Path(root_dir) / path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image_path = self.resolve_path(
            self.image_dir,
            row[self.image_col],
        )

        mask_value = (
            row[self.mask_col] if self.mask_col is not None else row[self.image_col]
        )

        mask_path = self.resolve_path(
            self.mask_dir,
            mask_value,
        )

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )
        mask_raw = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED,
        )

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(
                mask_raw,
                cv2.COLOR_BGR2RGB,
            )

        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )
            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }


def build_eval_transform(config):
    transform_steps = [
        A.Resize(
            height=int(config["image_height"]),
            width=int(config["image_width"]),
        )
    ]

    if config["normalize_input"]:
        transform_steps.append(
            A.Normalize(
                mean=tuple(config["input_mean"]),
                std=tuple(config["input_std"]),
            )
        )

    transform_steps.append(ToTensorV2())

    return A.Compose(transform_steps)


def build_test_data(config):
    device = torch.device(config["device"])
    eval_transform = build_eval_transform(config)

    dataset_root = Path(
        kagglehub.dataset_download(
            config["dataset_handle"], output_dir="../data/kagglehub"
        )
    )

    data_dir = dataset_root / "data"
    test_csv_path = data_dir / "sample" / "test.csv"

    if not test_csv_path.exists():
        raise FileNotFoundError(f"Test CSV not found: {test_csv_path}")

    test_df = pd.read_csv(test_csv_path)

    test_dataset = SegmentationEvaluationDataset(
        dataframe=test_df,
        image_dir=data_dir,
        mask_dir=data_dir,
        transform=eval_transform,
        image_col=config["image_col"],
        mask_col=config["mask_col"],
    )

    if len(test_dataset) == 0:
        raise ValueError("Test dataset is empty.")

    # Read one transformed item in the notebook process before evaluation.
    # This makes missing/corrupt image or mask errors show their real path.
    first_sample = test_dataset[0]
    expected_size = (
        int(config["image_height"]),
        int(config["image_width"]),
    )

    if tuple(first_sample["image"].shape[-2:]) != expected_size:
        raise ValueError(
            "Evaluation image transform returned size "
            f"{tuple(first_sample['image'].shape[-2:])}; "
            f"expected {expected_size}."
        )

    if tuple(first_sample["mask"].shape[-2:]) != expected_size:
        raise ValueError(
            "Evaluation mask transform returned size "
            f"{tuple(first_sample['mask'].shape[-2:])}; "
            f"expected {expected_size}."
        )

    num_workers = int(config.get("num_workers", 0))

    test_loader = DataLoader(
        test_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=num_workers,
        pin_memory=device.type == "cuda",
    )

    print(
        f"DataLoader workers: {num_workers} "
        "(use 0 for Windows/Jupyter)"
    )

    return test_df, test_dataset, test_loader

In [ ]:
"""
Swin Transformer V2 backbone for Swin-Unet.

The implementation follows Microsoft's official Swin Transformer V2 model:
https://github.com/microsoft/Swin-Transformer/blob/main/models/swin_transformer_v2.py

The defaults match SwinV2-T pretrained on ImageNet-1K:
    img_size = 256
    patch_size = 4
    embed_dim = 96
    depths = (2, 2, 6, 2)
    num_heads = (3, 6, 12, 24)
    window_size = 8

Expected checkpoint:
    swinv2_tiny_patch4_window8_256.pth

Microsoft Swin Transformer is released under the MIT License.
"""

from __future__ import annotations

import math
from typing import List, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

try:
    from timm.layers import DropPath, to_2tuple, trunc_normal_
except ImportError:
    from timm.models.layers import DropPath, to_2tuple, trunc_normal_


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int | None = None,
        out_features: int | None = None,
        act_layer=nn.GELU,
        drop: float = 0.0,
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """Partition ``[B, H, W, C]`` features into non-overlapping windows."""
    B, H, W, C = x.shape
    if H % window_size != 0 or W % window_size != 0:
        raise ValueError(
            f"Feature resolution ({H}, {W}) must be divisible by "
            f"window_size={window_size}."
        )

    x = x.view(
        B,
        H // window_size,
        window_size,
        W // window_size,
        window_size,
        C,
    )
    return (
        x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    )


def window_reverse(
    windows: torch.Tensor,
    window_size: int,
    H: int,
    W: int,
) -> torch.Tensor:
    """Reverse :func:`window_partition`."""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(
        B,
        H // window_size,
        W // window_size,
        window_size,
        window_size,
        -1,
    )
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    """Swin V2 window attention with cosine attention and continuous bias."""

    def __init__(
        self,
        dim: int,
        window_size: Tuple[int, int],
        num_heads: int,
        qkv_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        pretrained_window_size: int | Tuple[int, int] = 0,
    ):
        super().__init__()
        self.dim = dim
        self.window_size = to_2tuple(window_size)
        self.pretrained_window_size = to_2tuple(pretrained_window_size)
        self.num_heads = num_heads

        self.logit_scale = nn.Parameter(torch.log(10 * torch.ones((num_heads, 1, 1))))
        self.cpb_mlp = nn.Sequential(
            nn.Linear(2, 512, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_heads, bias=False),
        )

        relative_coords_h = torch.arange(
            -(self.window_size[0] - 1),
            self.window_size[0],
            dtype=torch.float32,
        )
        relative_coords_w = torch.arange(
            -(self.window_size[1] - 1),
            self.window_size[1],
            dtype=torch.float32,
        )
        relative_coords_table = (
            torch.stack(
                torch.meshgrid(relative_coords_h, relative_coords_w, indexing="ij")
            )
            .permute(1, 2, 0)
            .contiguous()
            .unsqueeze(0)
        )

        if self.pretrained_window_size[0] > 0:
            normalizer_h = max(self.pretrained_window_size[0] - 1, 1)
            normalizer_w = max(self.pretrained_window_size[1] - 1, 1)
        else:
            normalizer_h = max(self.window_size[0] - 1, 1)
            normalizer_w = max(self.window_size[1] - 1, 1)

        relative_coords_table[:, :, :, 0] /= normalizer_h
        relative_coords_table[:, :, :, 1] /= normalizer_w
        relative_coords_table *= 8
        relative_coords_table = (
            torch.sign(relative_coords_table)
            * torch.log2(torch.abs(relative_coords_table) + 1.0)
            / math.log2(8)
        )
        self.register_buffer("relative_coords_table", relative_coords_table)

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        self.register_buffer("relative_position_index", relative_coords.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(dim))
            self.v_bias = nn.Parameter(torch.zeros(dim))
        else:
            self.q_bias = None
            self.v_bias = None

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B_, N, C = x.shape

        qkv_bias = None
        if self.q_bias is not None:
            qkv_bias = torch.cat(
                (self.q_bias, torch.zeros_like(self.v_bias), self.v_bias)
            )

        qkv = F.linear(x, self.qkv.weight, qkv_bias)
        qkv = qkv.reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
        logit_scale = torch.clamp(self.logit_scale, max=math.log(100.0)).exp()
        attn = attn * logit_scale

        relative_position_bias_table = self.cpb_mlp(self.relative_coords_table).view(
            -1, self.num_heads
        )
        relative_position_bias = relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(N, N, -1)
        relative_position_bias = 16 * torch.sigmoid(
            relative_position_bias.permute(2, 0, 1).contiguous()
        )
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return self.proj_drop(x)


class SwinTransformerBlock(nn.Module):
    """Swin V2 W-MSA/SW-MSA block with residual post-normalization."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        num_heads: int,
        window_size: int = 8,
        shift_size: int = 0,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float = 0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
        pretrained_window_size: int = 0,
    ):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio

        if min(input_resolution) <= window_size:
            self.shift_size = 0
            self.window_size = min(input_resolution)
        if not 0 <= self.shift_size < self.window_size:
            raise ValueError("shift_size must satisfy 0 <= shift_size < window_size")

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim=dim,
            window_size=to_2tuple(self.window_size),
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
            pretrained_window_size=to_2tuple(pretrained_window_size),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(
            in_features=dim,
            hidden_features=int(dim * mlp_ratio),
            act_layer=act_layer,
            drop=drop,
        )

        if self.shift_size > 0:
            H, W = input_resolution
            img_mask = torch.zeros((1, H, W, 1))
            h_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            w_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            count = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = count
                    count += 1

            mask_windows = window_partition(img_mask, self.window_size)
            mask_windows = mask_windows.view(-1, self.window_size**2)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(
                attn_mask != 0, float(-100.0)
            ).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None
        self.register_buffer("attn_mask", attn_mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = self.input_resolution
        B, L, C = x.shape
        if L != H * W:
            raise ValueError(f"Input token length {L} does not match H*W={H*W}.")

        shortcut = x
        x = x.view(B, H, W, C)
        shifted_x = (
            torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )
            if self.shift_size > 0
            else x
        )

        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size**2, C)
        attn_windows = self.attn(x_windows, mask=self.attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)
        x = (
            torch.roll(
                shifted_x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )
            if self.shift_size > 0
            else shifted_x
        )
        x = x.view(B, H * W, C)

        x = shortcut + self.drop_path(self.norm1(x))
        x = x + self.drop_path(self.norm2(self.mlp(x)))
        return x


class PatchMerging(nn.Module):
    """Swin V2 downsampling: reduction first, then normalization."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(2 * dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = self.input_resolution
        B, L, C = x.shape
        if L != H * W:
            raise ValueError("Input feature has wrong token length.")
        if H % 2 != 0 or W % 2 != 0:
            raise ValueError(f"PatchMerging requires even resolution, got ({H}, {W}).")

        x = x.view(B, H, W, C)
        x = torch.cat(
            (
                x[:, 0::2, 0::2, :],
                x[:, 1::2, 0::2, :],
                x[:, 0::2, 1::2, :],
                x[:, 1::2, 1::2, :],
            ),
            dim=-1,
        ).view(B, -1, 4 * C)
        x = self.reduction(x)
        return self.norm(x)


class BasicLayer(nn.Module):
    """One hierarchical Swin Transformer V2 stage."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        depth: int,
        num_heads: int,
        window_size: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float | Sequence[float] = 0.0,
        norm_layer=nn.LayerNorm,
        downsample=None,
        use_checkpoint: bool = False,
        pretrained_window_size: int = 0,
    ):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        if isinstance(drop_path, Sequence):
            drop_path_values = list(drop_path)
        else:
            drop_path_values = [drop_path] * depth

        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if i % 2 == 0 else window_size // 2,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop,
                    attn_drop=attn_drop,
                    drop_path=drop_path_values[i],
                    norm_layer=norm_layer,
                    pretrained_window_size=pretrained_window_size,
                )
                for i in range(depth)
            ]
        )
        self.downsample = (
            downsample(input_resolution, dim=dim, norm_layer=norm_layer)
            if downsample is not None
            else None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def init_respostnorm(self) -> None:
        """Use the zero-init residual post-norm scheme from official Swin V2."""
        for block in self.blocks:
            nn.init.constant_(block.norm1.bias, 0)
            nn.init.constant_(block.norm1.weight, 0)
            nn.init.constant_(block.norm2.bias, 0)
            nn.init.constant_(block.norm2.weight, 0)


class PatchEmbed(nn.Module):
    """Convert an image to non-overlapping patch tokens."""

    def __init__(
        self,
        img_size: int | Tuple[int, int] = 256,
        patch_size: int | Tuple[int, int] = 4,
        in_chans: int = 3,
        embed_dim: int = 96,
        norm_layer=None,
    ):
        super().__init__()
        self.img_size = to_2tuple(img_size)
        self.patch_size = to_2tuple(patch_size)
        if any(
            image_size % patch != 0
            for image_size, patch in zip(self.img_size, self.patch_size)
        ):
            raise ValueError(
                f"Image size {self.img_size} must be divisible by "
                f"patch size {self.patch_size}."
            )

        self.patches_resolution = (
            self.img_size[0] // self.patch_size[0],
            self.img_size[1] // self.patch_size[1],
        )
        self.num_patches = self.patches_resolution[0] * self.patches_resolution[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )
        self.norm = norm_layer(embed_dim) if norm_layer is not None else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, _, H, W = x.shape
        if (H, W) != self.img_size:
            raise ValueError(f"Expected input size {self.img_size}, got ({H}, {W}).")
        x = self.proj(x).flatten(2).transpose(1, 2)
        return self.norm(x) if self.norm is not None else x


class SwinTransformerBackbone(nn.Module):
    """Hierarchical Swin Transformer V2 encoder for Swin-Unet."""

    def __init__(
        self,
        img_size: int = 256,
        patch_size: int = 4,
        in_chans: int = 3,
        embed_dim: int = 96,
        depths: Sequence[int] = (2, 2, 6, 2),
        num_heads: Sequence[int] = (3, 6, 12, 24),
        window_size: int = 8,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.2,
        norm_layer=nn.LayerNorm,
        ape: bool = False,
        patch_norm: bool = True,
        use_checkpoint: bool = False,
        pretrained_window_sizes: Sequence[int] = (0, 0, 0, 0),
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.depths = tuple(depths)
        self.num_heads = tuple(num_heads)
        self.window_size = window_size
        self.num_layers = len(depths)
        self.ape = ape
        self.patch_norm = patch_norm
        self.pretrained_window_sizes = tuple(pretrained_window_sizes)

        if len(num_heads) != self.num_layers:
            raise ValueError("num_heads and depths must have the same length.")
        if len(pretrained_window_sizes) != self.num_layers:
            raise ValueError(
                "pretrained_window_sizes must contain one value per encoder stage."
            )

        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.out_channels = tuple(int(embed_dim * 2**i) for i in range(self.num_layers))
        self.patch_embed = PatchEmbed(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            norm_layer=norm_layer if patch_norm else None,
        )
        num_patches = self.patch_embed.num_patches
        self.patches_resolution = self.patch_embed.patches_resolution

        total_downsample = 2 ** (self.num_layers - 1)
        if any(size % total_downsample != 0 for size in self.patches_resolution):
            raise ValueError(
                f"Patch resolution {self.patches_resolution} must be divisible by "
                f"{total_downsample} for {self.num_layers} stages."
            )

        if ape:
            self.absolute_pos_embed = nn.Parameter(
                torch.zeros(1, num_patches, embed_dim)
            )
            trunc_normal_(self.absolute_pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(p=drop_rate)

        dpr = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            resolution = (
                self.patches_resolution[0] // (2**i_layer),
                self.patches_resolution[1] // (2**i_layer),
            )
            effective_window = min(window_size, *resolution)
            if any(size % effective_window != 0 for size in resolution):
                raise ValueError(
                    f"Stage {i_layer} resolution {resolution} must be divisible "
                    f"by its effective window size {effective_window}."
                )

            self.layers.append(
                BasicLayer(
                    dim=int(embed_dim * 2**i_layer),
                    input_resolution=resolution,
                    depth=depths[i_layer],
                    num_heads=num_heads[i_layer],
                    window_size=window_size,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[sum(depths[:i_layer]) : sum(depths[: i_layer + 1])],
                    norm_layer=norm_layer,
                    downsample=PatchMerging if i_layer < self.num_layers - 1 else None,
                    use_checkpoint=use_checkpoint,
                    pretrained_window_size=pretrained_window_sizes[i_layer],
                )
            )

        self.norm = norm_layer(self.num_features)
        self.apply(self._init_weights)
        for layer in self.layers:
            layer.init_respostnorm()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self) -> set[str]:
        return {"absolute_pos_embed"}

    @torch.jit.ignore
    def no_weight_decay_keywords(self) -> set[str]:
        return {"cpb_mlp", "logit_scale", "relative_position_bias_table"}

    def forward(
        self,
        x: torch.Tensor,
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """Return the bottleneck tokens and the four encoder skip tensors."""
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        skips: List[torch.Tensor] = []
        for layer in self.layers:
            skips.append(x)
            x = layer(x)
        return self.norm(x), skips

    def load_pretrained(
        self,
        checkpoint_path: str,
        strict: bool = False,
    ):
        """Load an official Microsoft SwinV2 checkpoint into the encoder."""
        checkpoint_data = torch.load(checkpoint_path, map_location="cpu")
        state_dict = checkpoint_data.get("model", checkpoint_data)

        cleaned_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith("module."):
                key = key[len("module.") :]
            if key.startswith("backbone."):
                key = key[len("backbone.") :]
            if key.startswith("head."):
                continue
            if any(
                buffer_name in key
                for buffer_name in (
                    "relative_position_index",
                    "relative_coords_table",
                    "attn_mask",
                )
            ):
                continue
            cleaned_state_dict[key] = value

        if not any(key.endswith("attn.logit_scale") for key in cleaned_state_dict):
            raise ValueError(
                "Checkpoint is not Swin Transformer V2: attention logit_scale "
                "weights were not found."
            )

        model_state = self.state_dict()
        compatible_state = {}
        skipped = []
        for key, value in cleaned_state_dict.items():
            if key not in model_state:
                continue
            if model_state[key].shape != value.shape:
                skipped.append((key, tuple(value.shape), tuple(model_state[key].shape)))
                continue
            compatible_state[key] = value

        msg = self.load_state_dict(compatible_state, strict=strict)
        print(f"[SwinV2 backbone] Loaded {len(compatible_state)} tensors.")
        if skipped:
            print(
                f"[SwinV2 backbone] Skipped {len(skipped)} " "shape-mismatched tensors."
            )
        print(f"[SwinV2 backbone] Missing keys: {len(msg.missing_keys)}")
        print(f"[SwinV2 backbone] Unexpected keys: {len(msg.unexpected_keys)}")
        return msg

In [ ]:
"""
Swin-Unet model using a Swin Transformer V2 encoder and decoder.

Architecture adapted from the official Swin-Unet implementation:
https://github.com/HuCaoFighting/Swin-Unet

Encoder / pretrained weights:
https://github.com/microsoft/Swin-Transformer

The default model is Swin-Unet with a SwinV2-T ImageNet-1K encoder:
    img_size = 256
    patch_size = 4
    embed_dim = 96
    depths = (2, 2, 6, 2)
    num_heads = (3, 6, 12, 24)
    window_size = 8
"""

from __future__ import annotations

from typing import Sequence, Tuple

import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint

class PatchExpand(nn.Module):
    """2x token-space upsampling used by the Swin-Unet decoder."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        dim_scale: int = 2,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()

        self.input_resolution = input_resolution
        self.dim = dim
        self.dim_scale = dim_scale

        self.expand = (
            nn.Linear(
                dim,
                2 * dim,
                bias=False,
            )
            if dim_scale == 2
            else nn.Identity()
        )

        self.norm = norm_layer(dim // dim_scale)

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        H, W = self.input_resolution

        x = self.expand(x)

        B, L, C = x.shape

        if L != H * W:
            raise ValueError(f"Input token length {L} != H*W={H*W}.")

        x = x.view(B, H, W, C)

        x = x.view(
            B,
            H,
            W,
            2,
            2,
            C // 4,
        )

        x = (
            x.permute(
                0,
                1,
                3,
                2,
                4,
                5,
            )
            .contiguous()
            .view(
                B,
                H * 2,
                W * 2,
                C // 4,
            )
        )

        x = x.view(
            B,
            -1,
            C // 4,
        )

        x = self.norm(x)

        return x


class FinalPatchExpandX4(nn.Module):
    """Final 4x expansion from patch resolution back to image resolution."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        dim_scale: int = 4,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()

        self.input_resolution = input_resolution
        self.dim = dim
        self.dim_scale = dim_scale

        self.expand = nn.Linear(
            dim,
            (dim_scale**2) * dim,
            bias=False,
        )

        self.norm = norm_layer(dim)

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        H, W = self.input_resolution

        x = self.expand(x)

        B, L, C = x.shape

        if L != H * W:
            raise ValueError(f"Input token length {L} != H*W={H*W}.")

        scale = self.dim_scale

        x = x.view(
            B,
            H,
            W,
            scale,
            scale,
            C // (scale**2),
        )

        x = (
            x.permute(
                0,
                1,
                3,
                2,
                4,
                5,
            )
            .contiguous()
            .view(
                B,
                H * scale,
                W * scale,
                C // (scale**2),
            )
        )

        x = x.view(
            B,
            -1,
            self.dim,
        )

        x = self.norm(x)

        return x


class BasicLayerUp(nn.Module):
    """One Swin Transformer V2 decoder stage and optional PatchExpand."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        depth: int,
        num_heads: int,
        window_size: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float | Sequence[float] = 0.0,
        norm_layer=nn.LayerNorm,
        upsample=None,
        use_checkpoint: bool = False,
        pretrained_window_size: int = 0,
    ):
        super().__init__()

        self.use_checkpoint = use_checkpoint

        if isinstance(drop_path, Sequence):
            drop_path_values = list(drop_path)
        else:
            drop_path_values = [drop_path] * depth

        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=(0 if i % 2 == 0 else window_size // 2),
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop,
                    attn_drop=attn_drop,
                    drop_path=drop_path_values[i],
                    norm_layer=norm_layer,
                    pretrained_window_size=pretrained_window_size,
                )
                for i in range(depth)
            ]
        )

        if upsample is not None:
            self.upsample = upsample(
                input_resolution=input_resolution,
                dim=dim,
                dim_scale=2,
                norm_layer=norm_layer,
            )
        else:
            self.upsample = None

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = checkpoint.checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        if self.upsample is not None:
            x = self.upsample(x)

        return x

    def init_respostnorm(self) -> None:
        """Zero-init residual post-norm layers as in the official V2 model."""
        for block in self.blocks:
            nn.init.constant_(block.norm1.bias, 0)
            nn.init.constant_(block.norm1.weight, 0)
            nn.init.constant_(block.norm2.bias, 0)
            nn.init.constant_(block.norm2.weight, 0)


class SwinUNet(nn.Module):
    """
    Swin-Unet with a configurable Swin encoder.

    Defaults match SwinV2-T ImageNet-1K + 3-class semantic segmentation.

    Args:
        img_size:
            Fixed model input size. SwinV2-T ImageNet-1K uses 256.

        num_classes:
            Segmentation classes. Use 3 for:
            background, non-neoplastic, neoplastic.

        pretrained_path:
            Optional Microsoft SwinV2-T ImageNet-1K checkpoint path.

        pretrained_decoder:
            If True, encoder stage weights are mirrored into compatible
            decoder Swin blocks following the initialization idea used by
            the official Swin-Unet implementation.
    """

    def __init__(
        self,
        img_size: int = 256,
        patch_size: int = 4,
        in_channels: int = 3,
        num_classes: int = 3,
        embed_dim: int = 96,
        depths: Sequence[int] = (2, 2, 6, 2),
        num_heads: Sequence[int] = (3, 6, 12, 24),
        window_size: int = 8,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.2,
        norm_layer=nn.LayerNorm,
        ape: bool = False,
        patch_norm: bool = True,
        use_checkpoint: bool = False,
        pretrained_path: str | None = None,
        pretrained_decoder: bool = True,
        pretrained_window_sizes: Sequence[int] = (0, 0, 0, 0),
    ):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.depths = tuple(depths)
        self.num_heads = tuple(num_heads)
        self.window_size = window_size
        self.num_layers = len(depths)
        self.pretrained_window_sizes = tuple(pretrained_window_sizes)

        # Encoder: Microsoft Swin Transformer V2-compatible backbone.
        self.encoder = SwinTransformerBackbone(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_channels,
            embed_dim=embed_dim,
            depths=depths,
            num_heads=num_heads,
            window_size=window_size,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate,
            norm_layer=norm_layer,
            ape=ape,
            patch_norm=patch_norm,
            use_checkpoint=use_checkpoint,
            pretrained_window_sizes=pretrained_window_sizes,
        )

        patches_resolution = self.encoder.patches_resolution

        # Same stochastic-depth schedule as encoder.
        dpr = [
            x.item()
            for x in torch.linspace(
                0,
                drop_path_rate,
                sum(depths),
            )
        ]

        # Decoder
        self.layers_up = nn.ModuleList()
        self.concat_back_dim = nn.ModuleList()

        for i_layer in range(self.num_layers):
            encoder_stage = self.num_layers - 1 - i_layer

            dim = int(embed_dim * 2**encoder_stage)

            resolution = (
                patches_resolution[0] // (2**encoder_stage),
                patches_resolution[1] // (2**encoder_stage),
            )

            if i_layer == 0:
                self.concat_back_dim.append(nn.Identity())

                self.layers_up.append(
                    PatchExpand(
                        input_resolution=resolution,
                        dim=dim,
                        dim_scale=2,
                        norm_layer=norm_layer,
                    )
                )

            else:
                self.concat_back_dim.append(
                    nn.Linear(
                        2 * dim,
                        dim,
                    )
                )

                stage_dpr = dpr[
                    sum(depths[:encoder_stage]) : sum(depths[: encoder_stage + 1])
                ]

                self.layers_up.append(
                    BasicLayerUp(
                        dim=dim,
                        input_resolution=resolution,
                        depth=depths[encoder_stage],
                        num_heads=num_heads[encoder_stage],
                        window_size=window_size,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        drop_path=stage_dpr,
                        norm_layer=norm_layer,
                        upsample=(
                            PatchExpand if i_layer < self.num_layers - 1 else None
                        ),
                        use_checkpoint=use_checkpoint,
                        pretrained_window_size=pretrained_window_sizes[encoder_stage],
                    )
                )

        self.norm_up = norm_layer(embed_dim)

        self.final_up = FinalPatchExpandX4(
            input_resolution=patches_resolution,
            dim=embed_dim,
            dim_scale=4,
            norm_layer=norm_layer,
        )

        self.segmentation_head = nn.Conv2d(
            in_channels=embed_dim,
            out_channels=num_classes,
            kernel_size=1,
            bias=False,
        )

        # Keep the encoder's zero-initialized residual post-norm parameters.
        for module in (
            self.layers_up,
            self.concat_back_dim,
            self.norm_up,
            self.final_up,
            self.segmentation_head,
        ):
            module.apply(self._init_decoder_weights)
        for layer_up in self.layers_up:
            if isinstance(layer_up, BasicLayerUp):
                layer_up.init_respostnorm()

        if pretrained_path is not None:
            self.load_pretrained(
                pretrained_path,
                load_decoder=pretrained_decoder,
            )

    @staticmethod
    def _init_decoder_weights(
        module: nn.Module,
    ) -> None:
        try:
            from timm.layers import trunc_normal_
        except ImportError:
            from timm.models.layers import trunc_normal_

        if isinstance(module, nn.Linear):
            trunc_normal_(
                module.weight,
                std=0.02,
            )
            if module.bias is not None:
                nn.init.constant_(
                    module.bias,
                    0,
                )

        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(
                module.bias,
                0,
            )
            nn.init.constant_(
                module.weight,
                1.0,
            )

    def forward_up_features(
        self,
        x: torch.Tensor,
        skips: list[torch.Tensor],
    ) -> torch.Tensor:
        for i, layer_up in enumerate(self.layers_up):
            if i == 0:
                x = layer_up(x)
                continue

            skip_index = self.num_layers - 1 - i

            skip = skips[skip_index]

            if x.shape[1] != skip.shape[1]:
                raise ValueError(
                    f"Decoder/skip token mismatch at stage {i}: "
                    f"{x.shape} vs {skip.shape}"
                )

            x = torch.cat(
                [x, skip],
                dim=-1,
            )

            x = self.concat_back_dim[i](x)
            x = layer_up(x)

        x = self.norm_up(x)

        return x

    def tokens_to_feature_map(
        self,
        x: torch.Tensor,
        resolution: Tuple[int, int],
    ) -> torch.Tensor:
        B, L, C = x.shape
        H, W = resolution

        if L != H * W:
            raise ValueError(f"Token length {L} != H*W={H*W}.")

        return x.transpose(1, 2).contiguous().view(B, C, H, W)

    def forward(
        self,
        x: torch.Tensor,
        return_features: bool = False,
    ):
        if x.shape[1] != self.in_channels:
            raise ValueError(
                f"Expected {self.in_channels} input channels, " f"got {x.shape[1]}."
            )

        bottleneck, skips = self.encoder(x)

        decoder_tokens = self.forward_up_features(
            bottleneck,
            skips,
        )

        x = self.final_up(decoder_tokens)

        B, L, C = x.shape

        if L != self.img_size * self.img_size:
            raise ValueError(
                "Final token count does not match output image resolution."
            )

        x = x.view(
            B,
            self.img_size,
            self.img_size,
            C,
        )

        x = x.permute(
            0,
            3,
            1,
            2,
        ).contiguous()

        logits = self.segmentation_head(x)

        if not return_features:
            return logits

        bottleneck_resolution = (
            self.encoder.patches_resolution[0] // (2 ** (self.num_layers - 1)),
            self.encoder.patches_resolution[1] // (2 ** (self.num_layers - 1)),
        )

        bottleneck_map = self.tokens_to_feature_map(
            bottleneck,
            bottleneck_resolution,
        )

        decoder_map = self.tokens_to_feature_map(
            decoder_tokens,
            self.encoder.patches_resolution,
        )

        return logits, {
            "bottleneck": bottleneck_map,
            "decoder": decoder_map,
        }

    def load_pretrained(
        self,
        checkpoint_path: str,
        load_decoder: bool = True,
    ):
        """
        Load an official Microsoft SwinV2 checkpoint into the encoder.

        If load_decoder=True, compatible encoder Swin block weights are
        mirrored into decoder stages, following the official Swin-Unet
        pretraining strategy.
        """

        # Load exact Microsoft-compatible V2 keys into the encoder.
        encoder_msg = self.encoder.load_pretrained(
            checkpoint_path,
            strict=False,
        )

        if not load_decoder:
            return encoder_msg

        # Read checkpoint again to initialize compatible decoder blocks.
        checkpoint_data = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

        state_dict = checkpoint_data.get(
            "model",
            checkpoint_data,
        )

        cleaned = {}

        for key, value in state_dict.items():
            if key.startswith("module."):
                key = key[len("module.") :]

            if key.startswith("backbone."):
                key = key[len("backbone.") :]

            if key.startswith("head."):
                continue

            if any(
                buffer_name in key
                for buffer_name in (
                    "relative_position_index",
                    "relative_coords_table",
                    "attn_mask",
                )
            ):
                continue

            cleaned[key] = value

        model_state = self.state_dict()

        decoder_state = {}

        # Map encoder layer N -> decoder layer (3-N).
        # Only keys with exact matching names/shapes are loaded.
        for key, value in cleaned.items():
            if not key.startswith("layers."):
                continue

            parts = key.split(".")

            if len(parts) < 3:
                continue

            encoder_stage = int(parts[1])
            decoder_stage = self.num_layers - 1 - encoder_stage

            new_key = f"layers_up.{decoder_stage}." + ".".join(parts[2:])

            if new_key not in model_state:
                continue

            if model_state[new_key].shape != value.shape:
                continue

            decoder_state[new_key] = value

        decoder_msg = self.load_state_dict(
            decoder_state,
            strict=False,
        )

        print(
            f"[SwinV2-Unet decoder] Loaded " f"{len(decoder_state)} mirrored tensors."
        )

        return {
            "encoder": encoder_msg,
            "decoder": decoder_msg,
        }

In [ ]:
def load_checkpoint_file(
    checkpoint_path,
    map_location="cpu",
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {checkpoint_path}")

    checkpoint_data = torch.load(
        checkpoint_path,
        map_location=map_location,
        weights_only=False,
    )

    if not isinstance(checkpoint_data, dict):
        raise TypeError("Checkpoint must be a dictionary or raw state_dict.")

    return checkpoint_data


def is_raw_state_dict(value):
    return (
        isinstance(value, dict)
        and bool(value)
        and all(torch.is_tensor(item) for item in value.values())
    )


def extract_model_state_dict(checkpoint_data):
    if is_raw_state_dict(checkpoint_data):
        return checkpoint_data

    for key in (
        "model_state_dict",
        "state_dict",
        "model",
        "student_state_dict",
        "teacher_state_dict",
        "net",
        "network",
        "weights",
    ):
        candidate = checkpoint_data.get(key)

        if isinstance(candidate, nn.Module):
            return candidate.state_dict()

        if is_raw_state_dict(candidate):
            return candidate

        if isinstance(candidate, dict):
            for nested_key in (
                "state_dict",
                "model_state_dict",
                "weights",
            ):
                nested = candidate.get(nested_key)
                if is_raw_state_dict(nested):
                    return nested

    raise KeyError("Cannot find model weights in checkpoint.")


def strip_prefix(state_dict, prefix):
    return {
        (key[len(prefix) :] if key.startswith(prefix) else key): value
        for key, value in state_dict.items()
    }


def align_state_dict_keys(
    state_dict,
    model,
):
    model_state = model.state_dict()
    candidates = [state_dict]
    candidate = state_dict

    for _ in range(8):
        stripped = False

        for prefix in (
            "module.",
            "model.",
            "net.",
            "network.",
            "_orig_mod.",
        ):
            if candidate and all(key.startswith(prefix) for key in candidate):
                candidate = strip_prefix(
                    candidate,
                    prefix,
                )
                candidates.append(candidate)
                stripped = True
                break

        if not stripped:
            break

    def match_score(candidate_state):
        return sum(
            key in model_state and model_state[key].shape == value.shape
            for key, value in candidate_state.items()
        )

    return max(candidates, key=match_score)


def validate_checkpoint_config(
    checkpoint_data,
    model_label,
    config,
):
    saved_config = checkpoint_data.get("config")

    if not isinstance(saved_config, dict):
        return

    scalar_keys = (
        "image_height",
        "image_width",
        "num_classes",
        "img_size",
        "patch_size",
        "embed_dim",
        "window_size",
    )

    for key in scalar_keys:
        saved_value = saved_config.get(key)

        if saved_value is None:
            continue

        if int(saved_value) != int(config[key]):
            raise ValueError(
                f"{model_label}: checkpoint {key}={saved_value}, "
                f"but evaluation {key}={config[key]}."
            )

    saved_backbone = saved_config.get("backbone")
    if saved_backbone is not None and saved_backbone != config["backbone"]:
        raise ValueError(
            f"{model_label}: checkpoint backbone={saved_backbone!r}, "
            f"but evaluation expects {config['backbone']!r}."
        )

    sequence_keys = (
        "depths",
        "num_heads",
        "pretrained_window_sizes",
    )

    for key in sequence_keys:
        saved_value = saved_config.get(key)

        if saved_value is None:
            continue

        if tuple(saved_value) != tuple(config[key]):
            raise ValueError(
                f"{model_label}: checkpoint {key}={tuple(saved_value)}, "
                f"but evaluation {key}={tuple(config[key])}."
            )


def load_weights_into_swin_unet(
    model,
    checkpoint_data,
    model_label,
    strict=True,
):
    state_dict = align_state_dict_keys(
        extract_model_state_dict(checkpoint_data),
        model,
    )

    if not any(key.endswith(".attn.logit_scale") for key in state_dict):
        raise ValueError(
            f"{model_label}: checkpoint does not contain Swin Transformer V2 "
            "attention weights. A Swin V1 checkpoint cannot be evaluated with "
            "this architecture."
        )

    result = model.load_state_dict(
        state_dict,
        strict=bool(strict),
    )

    print(
        f"[{model_label}] "
        f"missing={len(result.missing_keys)} | "
        f"unexpected={len(result.unexpected_keys)}"
    )

    return result


def build_and_load_swin_unet(
    checkpoint_path,
    model_label,
    config,
):
    model = SwinUNet(
        img_size=int(config["img_size"]),
        patch_size=int(config["patch_size"]),
        in_channels=int(config["in_channels"]),
        num_classes=int(config["num_classes"]),
        embed_dim=int(config["embed_dim"]),
        depths=tuple(config["depths"]),
        num_heads=tuple(config["num_heads"]),
        window_size=int(config["window_size"]),
        mlp_ratio=float(config["mlp_ratio"]),
        qkv_bias=bool(config["qkv_bias"]),
        drop_rate=float(config["drop_rate"]),
        attn_drop_rate=float(config["attn_drop_rate"]),
        drop_path_rate=float(config["drop_path_rate"]),
        ape=bool(config["ape"]),
        patch_norm=bool(config["patch_norm"]),
        use_checkpoint=bool(config["use_checkpoint"]),
        pretrained_window_sizes=tuple(config["pretrained_window_sizes"]),
        pretrained_path=None,
        pretrained_decoder=False,
    )

    checkpoint_data = load_checkpoint_file(
        checkpoint_path,
        map_location="cpu",
    )

    validate_checkpoint_config(
        checkpoint_data=checkpoint_data,
        model_label=model_label,
        config=config,
    )

    load_weights_into_swin_unet(
        model=model,
        checkpoint_data=checkpoint_data,
        model_label=model_label,
        strict=config["strict_weight_loading"],
    )

    model.eval()

    return model

In [ ]:
def extract_swin_logits(
    model_output,
    model_label,
):
    if torch.is_tensor(model_output):
        logits = model_output

    elif isinstance(model_output, (tuple, list)):
        tensors = [item for item in model_output if torch.is_tensor(item)]
        logits = next(
            (item for item in tensors if item.ndim == 4),
            None,
        )

    elif isinstance(model_output, dict):
        logits = None

        for key in (
            "logits",
            "out",
            "prediction",
            "pred",
        ):
            value = model_output.get(key)

            if torch.is_tensor(value):
                logits = value
                break

    elif hasattr(model_output, "logits") and torch.is_tensor(model_output.logits):
        logits = model_output.logits

    else:
        logits = None

    if logits is None:
        raise TypeError(f"{model_label}: cannot extract logits " "from model output.")

    if logits.ndim != 4:
        raise ValueError(
            f"{model_label}: expected logits "
            f"[B, C, H, W], got {tuple(logits.shape)}."
        )

    return logits


@torch.inference_mode()
def predict_swin_batch(
    model,
    images,
    model_label,
    num_classes,
    output_size=None,
):
    logits = extract_swin_logits(
        model(images),
        model_label,
    )

    if logits.shape[1] != num_classes:
        raise ValueError(
            f"{model_label}: expected "
            f"{num_classes} output channels, "
            f"got {logits.shape[1]}."
        )

    if output_size is not None and logits.shape[-2:] != tuple(output_size):
        logits = F.interpolate(
            logits,
            size=tuple(output_size),
            mode="bilinear",
            align_corners=False,
        )

    return torch.argmax(
        logits,
        dim=1,
    )


def hd95_binary(pred_mask, true_mask):
    """
    Directional-max symmetric HD95 for one binary mask pair.

    The 95th percentile is computed separately in both directions and the
    larger value is returned, matching the non-directed MONAI/DeepMind style.

    Empty-mask convention:
        - both prediction and ground truth empty -> NaN (excluded from mean)
        - exactly one side empty                 -> image-diagonal penalty
    """
    pred = np.asarray(pred_mask, dtype=bool)
    true = np.asarray(true_mask, dtype=bool)

    pred_empty = not pred.any()
    true_empty = not true.any()

    if pred_empty and true_empty:
        return np.nan

    if pred_empty or true_empty:
        height, width = true.shape[-2:]
        return float(np.hypot(height, width))

    pred_surface = pred ^ binary_erosion(pred)
    true_surface = true ^ binary_erosion(true)

    if not pred_surface.any():
        pred_surface = pred
    if not true_surface.any():
        true_surface = true

    distance_to_true = distance_transform_edt(~true_surface)
    distance_to_pred = distance_transform_edt(~pred_surface)

    distances_pred_to_true = distance_to_true[pred_surface]
    distances_true_to_pred = distance_to_pred[true_surface]

    directed_hd95_pred_to_true = np.percentile(
        distances_pred_to_true,
        95,
    )
    directed_hd95_true_to_pred = np.percentile(
        distances_true_to_pred,
        95,
    )

    return float(
        max(
            directed_hd95_pred_to_true,
            directed_hd95_true_to_pred,
        )
    )


def safe_divide(
    numerator,
    denominator,
):
    if denominator == 0:
        return np.nan

    return float(numerator / denominator)


@torch.inference_mode()
def evaluate_swin_model(
    model,
    model_label,
    loader,
    device,
    num_classes,
    class_names,
):
    confusion = np.zeros(
        (num_classes, num_classes),
        dtype=np.int64,
    )

    hd95_values = {class_id: [] for class_id in range(num_classes)}

    model = model.to(device)
    model.eval()

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )
        targets = (
            batch["mask"]
            .to(
                device,
                non_blocking=True,
            )
            .long()
        )

        predictions = predict_swin_batch(
            model=model,
            images=images,
            model_label=model_label,
            num_classes=num_classes,
            output_size=targets.shape[-2:],
        )

        targets_np = targets.cpu().numpy()
        predictions_np = predictions.cpu().numpy()

        encoded = num_classes * targets_np.reshape(-1) + predictions_np.reshape(-1)

        confusion += np.bincount(
            encoded,
            minlength=num_classes**2,
        ).reshape(
            num_classes,
            num_classes,
        )

        for sample_index in range(targets_np.shape[0]):
            for class_id in range(num_classes):
                hd95_values[class_id].append(
                    hd95_binary(
                        predictions_np[sample_index] == class_id,
                        targets_np[sample_index] == class_id,
                    )
                )

    rows = []

    for class_id in range(num_classes):
        true_positive = int(
            confusion[
                class_id,
                class_id,
            ]
        )

        false_positive = int(confusion[:, class_id].sum() - true_positive)

        false_negative = int(confusion[class_id, :].sum() - true_positive)

        gt_pixels = int(confusion[class_id, :].sum())

        pred_pixels = int(
            confusion[
                :,
                class_id,
            ].sum()
        )

        hd_values = np.asarray(
            hd95_values[class_id],
            dtype=np.float64,
        )
        valid_hd_values = hd_values[~np.isnan(hd_values)]

        rows.append(
            {
                "model": model_label,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "dice": safe_divide(
                    2 * true_positive,
                    (2 * true_positive + false_positive + false_negative),
                ),
                "iou": safe_divide(
                    true_positive,
                    (true_positive + false_positive + false_negative),
                ),
                "hd95": (
                    float(valid_hd_values.mean())
                    if valid_hd_values.size > 0
                    else np.nan
                ),
                "precision": safe_divide(
                    true_positive,
                    (true_positive + false_positive),
                ),
                "recall": safe_divide(
                    true_positive,
                    (true_positive + false_negative),
                ),
                "gt_pixels": gt_pixels,
                "pred_pixels": pred_pixels,
            }
        )

    class_results = pd.DataFrame(rows)

    foreground = class_results[class_results["class_id"] > 0]

    metric_columns = [
        "dice",
        "iou",
        "hd95",
        "precision",
        "recall",
    ]

    def macro_row(
        class_id,
        class_name,
        frame,
    ):
        row = {
            "model": model_label,
            "class_id": class_id,
            "class_name": class_name,
            "gt_pixels": int(frame["gt_pixels"].sum()),
            "pred_pixels": int(frame["pred_pixels"].sum()),
        }

        row.update(
            {column: frame[column].mean(skipna=True) for column in metric_columns}
        )

        return row

    results = pd.concat(
        [
            class_results,
            pd.DataFrame(
                [
                    macro_row(
                        "macro_fg",
                        "macro_foreground_only",
                        foreground,
                    ),
                    macro_row(
                        "macro_all",
                        "macro_all_classes",
                        class_results,
                    ),
                ]
            ),
        ],
        ignore_index=True,
    )

    model.to("cpu")

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return results, confusion


def evaluate_swin_checkpoints(
    models,
    loader,
    device,
    num_classes,
    class_names,
):
    evaluation_tables = {}
    confusion_matrices = {}

    for model_label, model in models.items():
        print(f"Evaluating: {model_label}")

        model_results, model_confusion = evaluate_swin_model(
            model=model,
            model_label=model_label,
            loader=loader,
            device=device,
            num_classes=num_classes,
            class_names=class_names,
        )

        evaluation_tables[model_label] = model_results

        confusion_matrices[model_label] = model_confusion

    evaluation_df = pd.concat(
        evaluation_tables.values(),
        ignore_index=True,
    )

    model_comparison_df = evaluation_df[
        evaluation_df["class_id"].eq("macro_fg")
    ].reset_index(drop=True)

    return (
        evaluation_df,
        model_comparison_df,
        confusion_matrices,
    )

# Main


In [ ]:
CONFIG = copy.deepcopy(DEFAULT_EVAL_CONFIG)
device = torch.device(CONFIG["device"])

class_names = {
    class_id: BASE_CLASS_NAMES.get(
        class_id,
        f"class_{class_id}",
    )
    for class_id in range(int(CONFIG["num_classes"]))
}

test_df, test_dataset, test_loader = build_test_data(CONFIG)

print("Device      :", device)
print("Architecture: SwinV2-Unet (SwinV2-T, 256/window 8)")
print("Test samples:", len(test_dataset))

In [ ]:
# Update these filenames if the training notebook saved them with an epoch suffix.
CHECKPOINTS = {
    "swinv2_unet_half": r"B:\_projects\poply-image-segmentation\models_swinv2\best_SwinUNet_half.pth",
    "swinv2_unet_quarter": r"B:\_projects\poply-image-segmentation\models_swinv2\best_SwinUNet_quarter.pth",
}

SWIN_MODELS = {
    model_label: build_and_load_swin_unet(
        checkpoint_path=checkpoint_path,
        model_label=model_label,
        config=CONFIG,
    )
    for model_label, checkpoint_path in CHECKPOINTS.items()
}

print("Loaded models:", list(SWIN_MODELS))

In [ ]:
(
    evaluation_df,
    model_comparison_df,
    confusion_matrices,
) = evaluate_swin_checkpoints(
    models=SWIN_MODELS,
    loader=test_loader,
    device=device,
    num_classes=int(CONFIG["num_classes"]),
    class_names=class_names,
)

In [ ]:
print("\nPer-class evaluation:")
display(evaluation_df.round(4))

In [ ]:
print("\nMacro foreground comparison:")
display(model_comparison_df.round(4))

# Inference


In [ ]:
VIS_CLASS_COLORS = {
    0: (0, 0, 0),
    1: (0, 255, 0),
    2: (255, 0, 0),
}


def class_mask_to_rgb(class_mask, class_colors=VIS_CLASS_COLORS):
    """Convert class-index mask -> RGB mask."""
    class_mask = np.asarray(class_mask)

    rgb = np.zeros(
        (*class_mask.shape, 3),
        dtype=np.uint8,
    )

    for class_id, color in class_colors.items():
        rgb[class_mask == class_id] = color

    return rgb


def _read_rgb_image(image_path):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image_bgr = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    return cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB,
    )


def _read_gt_class_mask(ground_truth_path):
    ground_truth_path = Path(ground_truth_path)

    if not ground_truth_path.exists():
        raise FileNotFoundError(f"Ground truth not found: {ground_truth_path}")

    gt_raw = cv2.imread(
        str(ground_truth_path),
        cv2.IMREAD_UNCHANGED,
    )

    if gt_raw is None:
        raise FileNotFoundError(f"Cannot read ground truth: {ground_truth_path}")

    if gt_raw.ndim == 3:
        gt_raw = cv2.cvtColor(
            gt_raw,
            cv2.COLOR_BGR2RGB,
        )

    return decode_ground_truth(gt_raw)


@torch.inference_mode()
def inference_swin_image(
    model,
    model_label,
    image_path,
    transform,
    device,
    num_classes,
    restore_original_size=True,
):
    """
    Inference one image with the same Swin-UNet prediction path used by evaluation.

    Returns:
        pred_mask: HxW class-index mask.
    """
    model = model.to(device)
    model.eval()

    image_rgb = _read_rgb_image(image_path)
    original_height, original_width = image_rgb.shape[:2]

    transformed = transform(image=image_rgb)
    image_tensor = (
        transformed["image"]
        .unsqueeze(0)
        .to(
            device,
            non_blocking=True,
        )
    )

    pred_mask = (
        predict_swin_batch(
            model=model,
            images=image_tensor,
            model_label=model_label,
            num_classes=int(num_classes),
            output_size=None,
        )[0]
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    if restore_original_size:
        pred_mask = cv2.resize(
            pred_mask,
            (original_width, original_height),
            interpolation=cv2.INTER_NEAREST,
        )

    return pred_mask


def visualize_swin_inference_samples(
    model,
    model_label,
    dataset,
    device,
    num_classes,
    n_sample=4,
    alpha=0.40,
    seed=None,
):
    """
    Random n_sample from test dataset.

    Each image is displayed on ONE ROW:
        Original | Ground Truth | Prediction | Overlay
    """
    if n_sample <= 0:
        raise ValueError("n_sample must be greater than 0.")

    if len(dataset) == 0:
        raise ValueError("Dataset is empty.")

    if dataset.transform is None:
        raise ValueError("Dataset transform is required for inference.")

    n_sample = min(int(n_sample), len(dataset))

    if seed is None:
        sample_indices = np.random.choice(
            len(dataset),
            size=n_sample,
            replace=False,
        )
    else:
        rng = np.random.default_rng(int(seed))
        sample_indices = rng.choice(
            len(dataset),
            size=n_sample,
            replace=False,
        )

    fig, axes = plt.subplots(
        n_sample,
        4,
        figsize=(20, 5 * n_sample),
    )

    if n_sample == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, index in enumerate(sample_indices):
        sample = dataset[int(index)]
        image_path = Path(sample["image_path"])
        gt_path = Path(sample["mask_path"])

        pred_mask = inference_swin_image(
            model=model,
            model_label=model_label,
            image_path=image_path,
            transform=dataset.transform,
            device=device,
            num_classes=num_classes,
            restore_original_size=True,
        )

        image_rgb = _read_rgb_image(image_path)
        gt_class = _read_gt_class_mask(gt_path)

        gt_rgb = class_mask_to_rgb(gt_class)
        pred_rgb = class_mask_to_rgb(pred_mask)

        if gt_rgb.shape[:2] != image_rgb.shape[:2]:
            gt_rgb = cv2.resize(
                gt_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        if pred_rgb.shape[:2] != image_rgb.shape[:2]:
            pred_rgb = cv2.resize(
                pred_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        overlay = cv2.addWeighted(
            image_rgb,
            1.0 - float(alpha),
            pred_rgb,
            float(alpha),
            0,
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (gt_rgb, "Ground Truth"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]

        for col, (panel, title) in enumerate(panels):
            axes[row, col].imshow(panel)
            axes[row, col].set_title(title)
            axes[row, col].axis("off")

    fig.suptitle(
        f"Random test inference - {model_label}",
        fontsize=14,
        y=1.002,
    )
    plt.tight_layout(pad=0.5)
    plt.show()


def visualize_swin_inference_path(
    model,
    model_label,
    image_path,
    transform,
    device,
    num_classes,
    ground_truth_path=None,
    alpha=0.40,
):
    """
    Inference and visualize one explicitly specified image path.

    With ground_truth_path:
        Original | Ground Truth | Prediction | Overlay

    Without ground_truth_path:
        Original | Prediction | Overlay
    """
    image_path = Path(image_path)

    pred_mask = inference_swin_image(
        model=model,
        model_label=model_label,
        image_path=image_path,
        transform=transform,
        device=device,
        num_classes=num_classes,
        restore_original_size=True,
    )

    image_rgb = _read_rgb_image(image_path)
    pred_rgb = class_mask_to_rgb(pred_mask)

    if pred_rgb.shape[:2] != image_rgb.shape[:2]:
        pred_rgb = cv2.resize(
            pred_rgb,
            (image_rgb.shape[1], image_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        )

    overlay = cv2.addWeighted(
        image_rgb,
        1.0 - float(alpha),
        pred_rgb,
        float(alpha),
        0,
    )

    if ground_truth_path is None:
        _, axes = plt.subplots(
            1,
            3,
            figsize=(16, 5),
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]
    else:
        gt_class = _read_gt_class_mask(ground_truth_path)
        gt_rgb = class_mask_to_rgb(gt_class)

        if gt_rgb.shape[:2] != image_rgb.shape[:2]:
            gt_rgb = cv2.resize(
                gt_rgb,
                (image_rgb.shape[1], image_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        _, axes = plt.subplots(
            1,
            4,
            figsize=(20, 5),
        )

        panels = [
            (image_rgb, f"Original\n{image_path.name}"),
            (gt_rgb, "Ground Truth"),
            (pred_rgb, "Prediction"),
            (overlay, "Overlay"),
        ]

    for axis, (panel, title) in zip(axes, panels):
        axis.imshow(panel)
        axis.set_title(title)
        axis.axis("off")

    plt.tight_layout()
    plt.show()

    return pred_mask

## Random `n_sample` from test dataset


In [ ]:
VIS_MODEL_LABEL = next(iter(SWIN_MODELS))
VIS_MODEL = SWIN_MODELS[VIS_MODEL_LABEL]

N_SAMPLE = 4
VIS_SEED = None

visualize_swin_inference_samples(
    model=VIS_MODEL,
    model_label=VIS_MODEL_LABEL,
    dataset=test_dataset,
    device=device,
    num_classes=int(CONFIG["num_classes"]),
    n_sample=N_SAMPLE,
    alpha=0.40,
    seed=VIS_SEED,
)

## Inference by specified image path


In [ ]:
# Set a specific image path here.
IMAGE_PATH = None
GROUND_TRUTH_PATH = None  # optional

if IMAGE_PATH is None:
    print(
        "Set IMAGE_PATH to run path-based inference. "
        "Set GROUND_TRUTH_PATH too if you want the GT panel."
    )
else:
    visualize_swin_inference_path(
        model=VIS_MODEL,
        model_label=VIS_MODEL_LABEL,
        image_path=IMAGE_PATH,
        transform=test_dataset.transform,
        device=device,
        num_classes=int(CONFIG["num_classes"]),
        ground_truth_path=GROUND_TRUTH_PATH,
        alpha=0.40,
    )